In [1]:
from pathlib import Path

import confnotebook
from llama_cpp import Llama


In [2]:
source = Path("../examples/test/full")

files = sorted(source.glob("*.pdf"))
print(f"Found {len(files)} PDF files in {source}\n")
for i, file in enumerate(files):
    print(f"{i}: {file.name}")


Found 37 PDF files in ../examples/test/full

0: 10.pdf
1: 126164.pdf
2: 14964427_Енисейская ТГК-13-БРАЗ.pdf
3: 14976087_АвеларСолар Тех-БРАЗ-1.pdf
4: 15120979_Форвард Энерго-БРАЗ-1.pdf
5: 15235008_ОГК-2-БРАЗ-1.pdf
6: 25.pdf
7: 33.pdf
8: 4.pdf
9: 44.pdf
10: 7-1.pdf
11: АС ВНИИМ Менделеева Д.И. - БРАЗ на 31.12.24.pdf
12: АС ВНИИМ Менделеева Д.И. - РУ на 31.12.24.pdf
13: АС КРЕЗОЛ-САЗ на 31.08.25.pdf
14: АС Охрана Металлург-САЗ на 31.12.25.pdf
15: АС РУ- ВОСЬМОЙ ВЕТРОПАРК.pdf
16: АС Фрейт Линк-БРАЗ на 30.09.25.pdf
17: АСР СДД 2 кв.2024 (подп. к-а).pdf
18: Акт сверки взаимных расчетов №00000379931 от 30.04.2024.pdf
19: Акт сверки №0000.pdf
20: Акт сверки №MOW00-0087974   от 10.06.2024.pdf
21: Акт сверки №ТРБП-000006 от 10.01.2024.pdf
22: Браз-Юнигрин Пауэр.pdf
23: ЕВР-НКАЗ.pdf
24: Неформализованный_первичный_документ_23_ИИА_03_00342_от_31_03.pdf
25: Неформализованный_первичный_документ_23_ИИА_03_02596_от_31_03.pdf
26: Неформализованный_первичный_документ_23_ИИА_03_03623_от_31_03.pdf

In [3]:
IDX_FILE = 0

In [ ]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline_build = DocumentBuildPipeline()

document = pipeline_build.build(files[IDX_FILE].read_bytes())


/mnt/data/projects/rusal_recon_srv/repo/recon_vision/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
/mnt/data/projects/rusal_recon_srv/repo/recon_vision/.venv/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/PP-OCRv5_server_det')
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', '/mnt/data/projects/rusal_recon

In [5]:
def get_cell_covering(table, row: int, col: int):
    for cell in table.get_rows()[row]:
        if cell.col <= col < cell.col + cell.colspan:
            return cell
    return None


summary_text = ""
summary_cell_text: list[str] = []
for page in document.pages:
    text_paragraph = " ".join(paragraph.text for paragraph in page.paragraphs)
    for table in page.tables:
        if table.continuation_of is not None:
            continue
        dc = table.dc_cols
        num_row = table.get_dc_header_row()
        if num_row == -1 or not dc:
            continue
        seen_cells: set[int] = set()
        for i, col in enumerate(sorted(dc)):
            for j in range(num_row):
                cell = get_cell_covering(table, j, col)
                if cell is None or id(cell) in seen_cells:
                    continue
                seen_cells.add(id(cell))
                cell_text = cell.value.strip()
                if cell_text:
                    summary_cell_text.append(cell_text)

    summary_text += text_paragraph + " "

print("SUMMARY CELL TEXT:")
print(summary_cell_text)
print("\nSUMMARY TEXT:")
print(summary_text)

SUMMARY CELL TEXT:
['По данным Продавца', 'По данным Покупателя']

SUMMARY TEXT:
10 АКТ СВЕРКИ РАСЧЕТОВ между АО "РИР" и АО "РУСАЛ Урал" за период с 01.04.2025 по 30.06.2025 года по договору купли-продажи мощности по результатам конкурентного отбора мощности № КOМ-30018731-ОВТEPENG-VOLГOГAL-25-VV-1 от 02.09.2021 (py6.) От АO "РИР" Ведущий спецналнст ОРЭМ Санникова .Н. дов. № 307/261-ДОВ от 27.06.2024г./ dllam От AO "PУСAЛ Урал" /. Ф-ЛАС«ГИ»ВГОЗЕРСКЕ 456796ЧЕЛЯБИНСКАЯОБЛ. Г.ОЗЕРСК П.НОВОГОРНЫЙ УЛ. ЛЕНИНА1 АЯ 218 


In [ ]:
# сформируем для теста список текста из ячеек и параграфов из всех файлов


TEST_TEXT 

In [13]:
import json
import re

B_SIZE = 2

class CleanTextPipeline:

    def __init__(self):

        self.ollama = Llama(
            model_path=f"../models/Qwen3.5-{B_SIZE}B/Qwen3.5-{B_SIZE}B-PRISM-DQ.gguf",
            n_ctx=16834,
            n_threads=8,
            max_tokens=128,
            verbose=False
        )
        self.system_prompt = """
        Ты - ассистент, который помогает исправить ошибки распознавания текста из PDF документов.
        На вход тебе придет текст, который был распознан из документа. 
        Твоя задача - исправить ошибки распознавания, такие как лишние символы, неправильные пробелы и другие опечатки.
        """.strip()


    def run(self, text: str) -> str:
        user_prompt = "Исправь ошибки распознавания текста:\n\n" + "\n".join(text)
        response = self._response_llm(user_prompt)
        choices = response["choices"][0]["message"]["content"]

        return choices.strip()

    def _response_llm(self, user_prompt: str = None) -> dict:
        response = self.ollama .create_chat_completion(
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": user_prompt}
                ],
            response_format={
                "type": "json_object",
                "schema": {
                    "type": "object",
                    "properties": {
                        "text": {"type": "string"},
                    },
                    "required": ["text"],
                },
            },
            temperature=1.0,
            top_k=64,
            top_p=0.95,
            max_tokens=256,
            stream=False,
        )
        return response

In [14]:
cleaner = CleanTextPipeline()
cleaned_text = cleaner.run(summary_text)
print("CLEANED TEXT:")
print(cleaned_text)

llama_context: n_ctx_seq (16896) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


CLEANED TEXT:
{"text": "1 0 \\n\nА К Т \\n\nС В Е Р К И \\n\nР А С Ч Е Т О В \\n\nм еж д у \\n\nА О \\n\n\" Р И Р \" \\n\nи \\n\nА О \\n\n\" Р У С А Л \\n\nУ р а л \" \\n\nз а \\n\nп е р и о д \\n\nс \\n\n0 1 . 0 4 . 2 0 2 5 \\n\nп о \\n\n3 0 . 0 6 . 2 0 2 5 \\n\nг о д а \\n\nп о д о г о в о р у \\n\nк у п л и - п р о д а ж и \\n\nм о щ н о с т и \\n\nп о \\n\nр е з у л \" т а т а м \\n\nк о н к у р е н т н о г о \\n\nо т б о р а \\n\nм о щ н о с т и \\n\n№ \\n\nК О М -


In [11]:
ground_truth = [
    {"seller": "РИР, АО",                                              "buyer": "РУСАЛ УРАЛ, АО"},                                      # [0] 10.pdf
    {"seller": "БОГУЧАНСКАЯ ГЭС, АО",                                  "buyer": "РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},            # [1] 126164.pdf
    {"seller": "УНИИМ - ФИЛИАЛ ФГУП ВНИИМ ИМ. Д.И.МЕНДЕЛЕЕВА",         "buyer": "РУСАЛ КАНДАЛАКША,"},                                   # [2]
    {"seller": "АВЕЛАР СОЛАР ТЕХНОЛОДЖИ, ООО",                         "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [3]
    {"seller": "ФОРВАРД ЭНЕРГО, ПАО",                                  "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [4]
    {"seller": "ОГК-2, ПАО",                                           "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [5]
    {"seller": "ЦФР, АО",                                              "buyer": "РУСАЛ УРАЛ, АО"},                                      # [6]
    {"seller": "ТГК-1, ПАО",                                           "buyer": "РУСАЛ УРАЛ, АО"},                                      # [7]
    {"seller": "ЧЕТВЕРТЫЙ ВЕТРОПАРК, ООО",                             "buyer": "РУСАЛ УРАЛ, АО"},                                      # [8]
    {"seller": "ЦФР, АО",                                              "buyer": "РУСАЛ САЯНОГОРСК, АО"},                               # [9]
    {"seller": "ОГК-2, ПАО",                                           "buyer": "ОБЪЕДИНЕННАЯ КОМПАНИЯ РУСАЛ УРАЛЬСКИЙ АЛЮМИНИЙ, АО"},  # [10]
    {"seller": "ВНИИМ ИМ. Д.И. МЕНДЕЛЕЕВА, ФГУП",                      "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [11]
    {"seller": "ВНИИМ ИМ. Д.И. МЕНДЕЛЕЕВА, ФГУП",                      "buyer": "РУСАЛ УРАЛ, АО"},                                      # [12]
    {"seller": "ТД КРЕЗОЛ, ООО",                                       "buyer": "РУСАЛ САЯНОГОРСК, АО"},                                # [13]
    {"seller": "ОХРАНА МЕТАЛЛУРГ, ООО",                                "buyer": "РУСАЛ САЯНОГОРСК, АО"},                                # [14]
    {"seller": "ВОСЬМОЙ ВЕТРОПАРК, ООО",                               "buyer": "РУСАЛ УРАЛ, АО"},                                      # [15]
    {"seller": "ФРЕЙТ ЛИНК, АО",                                       "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [16]
    {"seller": "ФЕДЕРАЛЬНАЯ ГИДРОГЕНЕРИРУЮЩАЯ КОМПАНИЯ - РУСГИДРО, ПАО", "buyer": "ОБЪЕДИНЕННАЯ КОМПАНИЯ РУСАЛ УРАЛЬСКИЙ АЛЮМИНИЙ, АО"}, # [17]
    {"seller": "РН-КАРТ, ООО",                                         "buyer": "РУСАЛ АЧИНСКИЙ ГЛИНОЗЕМНЫЙ КОМБИНАТ, АО"},             # [18]
    {"seller": "НОВАЯ ЭНЕРГОСБЫТОВАЯ КОМПАНИЯ, ООО",                   "buyer": "РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},            # [19]
    {"seller": "ФРЕЙТ ЛИНК, АО",                                       "buyer": "ОК РУСАЛ ТД, АО"},                                     # [20]
    {"seller": "ТРАНСКОМ, ООО",                                        "buyer": "РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},            # [21]
    {"seller": "ЮНИГРИН ПАУЭР, ООО",                                   "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [22]
    {"seller": "ЕВРОСИБЭНЕРГО, АО",                                    "buyer": "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},           # [23]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "БОГУЧАНСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},                   # [24]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "БОГУЧАНСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},                   # [25]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},           # [26]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "БОГУЧАНСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},                   # [27]
    {"seller": "ТД КРЕЗОЛ, ООО",                                       "buyer": "РУСАЛ САЯНОГОРСК, АО"},                                # [28]
    {"seller": "ФРЕЙТ ЛИНК, АО",                                       "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [29]
    {"seller": "РЖД, ОАО",                                             "buyer": "РУСАЛ АЧИНСКИЙ ГЛИНОЗЕМНЫЙ КОМБИНАТ, АО"},             # [30]
    {"seller": "РИР, АО",                                              "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [31]
    {"seller": "РИР, АО",                                              "buyer": "РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},            # [32]
    {"seller": "СОЛНЕЧНЫЙ ВЕТЕР, АО",                                  "buyer": "ОБЪЕДИНЕННАЯ КОМПАНИЯ РУСАЛ УРАЛЬСКИЙ АЛЮМИНИЙ, АО"},  # [33]
    {"seller": "ТГК-2, ПАО",                                           "buyer": "РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},            # [34]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},           # [35]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},           # [36]
]

In [26]:


results = []

for i, file in enumerate(files):
    print(f"Processing file: {file.name}")
    pdf_bytes = file.read_bytes()
    try:
        document = pipeline_build.build(pdf_bytes)
    except Exception as e:
        print(f"Error building document for {file.name}: {e}")
        continue

    text = " ".join(paragraph.text for paragraph in document.pages[0].paragraphs)
    dict = {
        "id": i,
        "file": file.stem,
        "text": text,
        "seller": ground_truth[i]["seller"],
        "buyer": ground_truth[i]["buyer"]
    }
    # seller, buyer = pipeline_ext.run(document)
    # result = {
    #     "id": i,
    #     "seller": seller,
    #     "buyer": buyer
    # }
    results.append(dict)

2026-04-09 15:32:05.023 | INFO     | vision_core.pipelines.build_document:build:71 - Обработка страницы 0 с dpi 200...
2026-04-09 15:32:05.159 | INFO     | vision_core.pipelines.build_document:_process_page:117 - Коррекция ориентации и наклона...
2026-04-09 15:32:05.173 | DEBUG    | vision_core.preprocessor.image_orientation:process:40 - Ориентация страницы: 0° с точностью 0.9173


Processing file: 10.pdf


2026-04-09 15:32:09.087 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:123 - edge rotations: [-0.1, 13.9, 0.0, -0.35]°
2026-04-09 15:32:09.088 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:132 - после фильтрации: [-0.1, 0.0, -0.35]°
2026-04-09 15:32:09.088 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:140 - исключены нулевые углы из усреднения: [0.0]°
2026-04-09 15:32:09.089 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:146 - average: -0.225°  deviation: 0.177°
2026-04-09 15:32:09.091 | DEBUG    | vision_core.preprocessor.image_orientation:process:53 - Угол наклона страницы: -0.225°
2026-04-09 15:32:09.092 | INFO     | vision_core.pipelines.build_document:_process_page:119 - Коррекция завершена.
2026-04-09 15:32:09.092 | INFO     | vision_core.pipelines.build_document:_process_page:121 - Предобработка изображения...
2026-04-09 15:32:09.093 | INFO     | vision_core.pipelines.build_

Processing file: 126164.pdf


2026-04-09 15:32:12.897 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:123 - edge rotations: [0.0, 14.8, 0.0, -11.45]°
2026-04-09 15:32:12.897 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:132 - после фильтрации: [0.0, 0.0, -11.45]°
2026-04-09 15:32:12.898 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:140 - исключены нулевые углы из усреднения: [0.0, 0.0]°
2026-04-09 15:32:12.898 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:146 - average: -11.450°  deviation: 0.000°
2026-04-09 15:32:12.901 | DEBUG    | vision_core.preprocessor.image_orientation:process:53 - Угол наклона страницы: -11.450°
2026-04-09 15:32:12.901 | INFO     | vision_core.pipelines.build_document:_process_page:119 - Коррекция завершена.
2026-04-09 15:32:12.902 | INFO     | vision_core.pipelines.build_document:_process_page:121 - Предобработка изображения...
2026-04-09 15:32:12.902 | INFO     | vision_core.pipelines

...

2026-04-09 15:38:18.205 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:123 - edge rotations: [0.0, 12.7, 0.2, -9.2]°
2026-04-09 15:38:18.206 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:132 - после фильтрации: [0.0, 0.2, -9.2]°
2026-04-09 15:38:18.206 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:140 - исключены нулевые углы из усреднения: [0.0]°
2026-04-09 15:38:18.207 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:146 - average: -4.500°  deviation: 6.647°
2026-04-09 15:38:18.207 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:151 - deviation too large -- no rotation
2026-04-09 15:38:18.208 | DEBUG    | vision_core.preprocessor.image_orientation:process:53 - Угол наклона страницы: 0.000°
2026-04-09 15:38:18.208 | INFO     | vision_core.pipelines.build_document:_process_page:119 - Коррекция завершена.
2026-04-09 15:38:18.209 | INFO     | vision_core.pipeli

Processing file: документ 23-ИИА-03-01141 от 31_03_2025.pdf


2026-04-09 15:38:22.992 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:123 - edge rotations: [0.0, -0.0, 0.0, -0.0]°
2026-04-09 15:38:22.992 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:132 - после фильтрации: [0.0, -0.0, 0.0, -0.0]°
2026-04-09 15:38:22.993 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:146 - average: 0.000°  deviation: 0.000°
2026-04-09 15:38:22.994 | DEBUG    | vision_core.preprocessor.image_orientation:process:53 - Угол наклона страницы: 0.000°
2026-04-09 15:38:22.994 | INFO     | vision_core.pipelines.build_document:_process_page:119 - Коррекция завершена.
2026-04-09 15:38:22.994 | INFO     | vision_core.pipelines.build_document:_process_page:121 - Предобработка изображения...
2026-04-09 15:38:22.995 | INFO     | vision_core.pipelines.build_document:_process_page:123 - Предобработка завершена.
2026-04-09 15:38:22.995 | INFO     | vision_core.pipelines.build_document:_process_page:125 - 

Processing file: документ 23-ИИА-03-01142 от 31_03_2025.pdf


2026-04-09 15:38:44.832 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:123 - edge rotations: [0.0, -0.0, 0.0, -0.0]°
2026-04-09 15:38:44.832 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:132 - после фильтрации: [0.0, -0.0, 0.0, -0.0]°
2026-04-09 15:38:44.833 | DEBUG    | vision_core.preprocessor.image_orientation:_detect_rotation:146 - average: 0.000°  deviation: 0.000°
2026-04-09 15:38:44.834 | DEBUG    | vision_core.preprocessor.image_orientation:process:53 - Угол наклона страницы: 0.000°
2026-04-09 15:38:44.834 | INFO     | vision_core.pipelines.build_document:_process_page:119 - Коррекция завершена.
2026-04-09 15:38:44.835 | INFO     | vision_core.pipelines.build_document:_process_page:121 - Предобработка изображения...
2026-04-09 15:38:44.835 | INFO     | vision_core.pipelines.build_document:_process_page:123 - Предобработка завершена.
2026-04-09 15:38:44.836 | INFO     | vision_core.pipelines.build_document:_process_page:125 - 

In [27]:
# save results to file json
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)


In [13]:
for res in results:
    print(f"[{res['id']}] Seller: {res['seller']} - Buyer: {res['buyer']}\n")

[0] Seller: РИР, АО - Buyer: РУСАЛ УРАЛ, АО

[2] Seller: УНИИМ - ФИЛИАЛ ФГУП ВНИИМ ИМ. Д.И. МЕНДЕЛЕЕВА - Buyer: РУСАЛ КАНДАША

[3] Seller: АВЕЛАР СОЛАР ТЕХНОЛОДЖИ, ПАО - Buyer: РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО

[4] Seller: ФОРВАРД ЭКЕРГО, ПАО - Buyer: РУСАЛ БРАТСК, ПАО

[5] Seller: ОГК-2, ПАО - Buyer: РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО

[6] Seller: ЦФР, АО - Buyer: РУСАЛ УРАЛ, АО

[7] Seller: ТГК-1, ПАО - Buyer: РУСАЛ УРАЛ, АО

[8] Seller: ОБЩЕСТВЕННО ОГРАНИЧЕННАЯ ОТВЕТСТВЕННОСТЬ ЧЕТВЕРТЫЙ ВЕТРОПАРК ФРВ - Buyer: РУСАЛ УРАЛ, ООО

[9] Seller: ЦФР, АО - Buyer: РУСАЛ САЯНОГОРСК, АО

[10] Seller: ОГК-2, ПАО - Buyer: АХТОВСЕРК

[11] Seller: УНИИМ - ФИЛИАЛ ФГУП ВНИИМ ИМ. Д. И. МЕНДЕЛЕЕВА - Buyer: РУСАЛ БРАТСК, ПАО

[13] Seller: ТД КРЕЗОЛ, ООО - Buyer: РУСАЛ САЯНОГОРСК, АО

[14] Seller: ОХРАНА МЕТАЛЛУРГ, ООО - Buyer: РУСАЛ САЯНСКОГОРСК, АО

[15] Seller: ВОСЬМОЙ ВЕТРОПАРК ФР, ООО - Buyer: РУСАЛ УРАЛ, АО

[16] Seller: ФРЕЙТ ЛИНК, АО - Buyer: РУСАЛ БРАТСК, АО

[17] Seller: ФЕДЕРАЛЬНАЯ ГИД

In [14]:
# None = пропустить этот файл (плохое качество / неопределённо)

ground_truth = [
    {"seller": "РИР, АО",                                              "buyer": "РУСАЛ УРАЛ, АО"},                                      # [0] 10.pdf
    {"seller": "БОГУЧАНСКАЯ ГЭС, АО",                                  "buyer": "РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},            # [1] 126164.pdf
    {"seller": "УНИИМ - ФИЛИАЛ ФГУП ВНИИМ ИМ. Д.И.МЕНДЕЛЕЕВА",         "buyer": "РУСАЛ КАНДАЛАКША,"},                                   # [2]
    {"seller": "АВЕЛАР СОЛАР ТЕХНОЛОДЖИ, ООО",                         "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [3]
    {"seller": "ФОРВАРД ЭНЕРГО, ПАО",                                  "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [4]
    {"seller": "ОГК-2, ПАО",                                           "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [5]
    {"seller": "ЦФР, АО",                                              "buyer": "РУСАЛ УРАЛ, АО"},                                      # [6]
    {"seller": "ТГК-1, ПАО",                                           "buyer": "РУСАЛ УРАЛ, АО"},                                      # [7]
    {"seller": "ЧЕТВЕРТЫЙ ВЕТРОПАРК, ООО",                             "buyer": "РУСАЛ УРАЛ, АО"},                                      # [8]
    {"seller": "ЦФР, АО",                                              "buyer": "РУСАЛ САЯНОГОРСК, АО"},                               # [9]
    {"seller": "ОГК-2, ПАО",                                           "buyer": "ОБЪЕДИНЕННАЯ КОМПАНИЯ РУСАЛ УРАЛЬСКИЙ АЛЮМИНИЙ, АО"},  # [10]
    {"seller": "ВНИИМ ИМ. Д.И. МЕНДЕЛЕЕВА, ФГУП",                      "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [11]
    {"seller": "ВНИИМ ИМ. Д.И. МЕНДЕЛЕЕВА, ФГУП",                      "buyer": "РУСАЛ УРАЛ, АО"},                                      # [12]
    {"seller": "ТД КРЕЗОЛ, ООО",                                       "buyer": "РУСАЛ САЯНОГОРСК, АО"},                                # [13]
    {"seller": "ОХРАНА МЕТАЛЛУРГ, ООО",                                "buyer": "РУСАЛ САЯНОГОРСК, АО"},                                # [14]
    {"seller": "ВОСЬМОЙ ВЕТРОПАРК, ООО",                               "buyer": "РУСАЛ УРАЛ, АО"},                                      # [15]
    {"seller": "ФРЕЙТ ЛИНК, АО",                                       "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [16]
    {"seller": "ФЕДЕРАЛЬНАЯ ГИДРОГЕНЕРИРУЮЩАЯ КОМПАНИЯ - РУСГИДРО, ПАО", "buyer": "ОБЪЕДИНЕННАЯ КОМПАНИЯ РУСАЛ УРАЛЬСКИЙ АЛЮМИНИЙ, АО"}, # [17]
    {"seller": "РН-КАРТ, ООО",                                         "buyer": "РУСАЛ АЧИНСКИЙ ГЛИНОЗЕМНЫЙ КОМБИНАТ, АО"},             # [18]
    {"seller": "НОВАЯ ЭНЕРГОСБЫТОВАЯ КОМПАНИЯ, ООО",                   "buyer": "РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},            # [19]
    {"seller": "ФРЕЙТ ЛИНК, АО",                                       "buyer": "ОК РУСАЛ ТД, АО"},                                     # [20]
    {"seller": "ТРАНСКОМ, ООО",                                        "buyer": "РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},            # [21]
    {"seller": "ЮНИГРИН ПАУЭР, ООО",                                   "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [22]
    {"seller": "ЕВРОСИБЭНЕРГО, АО",                                    "buyer": "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},           # [23]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "БОГУЧАНСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},                   # [24]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "БОГУЧАНСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},                   # [25]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},           # [26]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "БОГУЧАНСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},                   # [27]
    {"seller": "ТД КРЕЗОЛ, ООО",                                       "buyer": "РУСАЛ САЯНОГОРСК, АО"},                                # [28]
    {"seller": "ФРЕЙТ ЛИНК, АО",                                       "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [29]
    {"seller": "РЖД, ОАО",                                             "buyer": "РУСАЛ АЧИНСКИЙ ГЛИНОЗЕМНЫЙ КОМБИНАТ, АО"},             # [30]
    {"seller": "РИР, АО",                                              "buyer": "РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО"},               # [31]
    {"seller": "РИР, АО",                                              "buyer": "РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},            # [32]
    {"seller": "СОЛНЕЧНЫЙ ВЕТЕР, АО",                                  "buyer": "ОБЪЕДИНЕННАЯ КОМПАНИЯ РУСАЛ УРАЛЬСКИЙ АЛЮМИНИЙ, АО"},  # [33]
    {"seller": "ТГК-2, ПАО",                                           "buyer": "РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},            # [34]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},           # [35]
    {"seller": "ИНТЕР РАО - ЭЛЕКТРОГЕНЕРАЦИЯ, АО",                     "buyer": "РУСАЛ НОВОКУЗНЕЦКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО"},           # [36]
]


In [15]:
# Сравниваем results с ground truth
correct = skipped = errors = 0

for i, (res, gt) in enumerate(zip(results, ground_truth, strict=False)):
    if gt is None:
        skipped += 1
        continue
    seller_ok = res["seller"] == gt["seller"]
    buyer_ok  = res["buyer"]  == gt["buyer"]
    if seller_ok and buyer_ok:
        correct += 1
    else:
        errors += 1
        print(f"[{i}] {files[i].name}")
        if not seller_ok:
            print(f"  seller: got={res['seller']!r}  expected={gt['seller']!r}")
        if not buyer_ok:
            print(f"  buyer:  got={res['buyer']!r}  expected={gt['buyer']!r}")

total = len(results) - skipped
print(f"\n{correct}/{total} correct, {skipped} skipped")


[1] 126164.pdf
  seller: got='УНИИМ - ФИЛИАЛ ФГУП ВНИИМ ИМ. Д.И. МЕНДЕЛЕЕВА'  expected='БОГУЧАНСКАЯ ГЭС, АО'
  buyer:  got='РУСАЛ КАНДАША'  expected='РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО'
[2] 14964427_Енисейская ТГК-13-БРАЗ.pdf
  seller: got='АВЕЛАР СОЛАР ТЕХНОЛОДЖИ, ПАО'  expected='УНИИМ - ФИЛИАЛ ФГУП ВНИИМ ИМ. Д.И.МЕНДЕЛЕЕВА'
  buyer:  got='РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО'  expected='РУСАЛ КАНДАЛАКША,'
[3] 14976087_АвеларСолар Тех-БРАЗ-1.pdf
  seller: got='ФОРВАРД ЭКЕРГО, ПАО'  expected='АВЕЛАР СОЛАР ТЕХНОЛОДЖИ, ООО'
  buyer:  got='РУСАЛ БРАТСК, ПАО'  expected='РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО'
[4] 15120979_Форвард Энерго-БРАЗ-1.pdf
  seller: got='ОГК-2, ПАО'  expected='ФОРВАРД ЭНЕРГО, ПАО'
[5] 15235008_ОГК-2-БРАЗ-1.pdf
  seller: got='ЦФР, АО'  expected='ОГК-2, ПАО'
  buyer:  got='РУСАЛ УРАЛ, АО'  expected='РУСАЛ БРАТСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, ПАО'
[6] 25.pdf
  seller: got='ТГК-1, ПАО'  expected='ЦФР, АО'
[7] 33.pdf
  seller: got='ОБЩЕСТВЕННО ОГРАНИЧЕННАЯ ОТВЕТСТВЕННОСТЬ